In [ ]:
import numpy as np
import os
import sys
import subprocess
import matplotlib.pyplot as plt
import glob

from multiprocessing import Pool

GMX_BIN_DIR = "/usr/local/gromacs/bin/gmx"
PLUMED_KERNEL_PATH = "/Users/wuzhiyou/plumed/lib/libplumedKernel.dylib"
PLUMED_BIN = "/Users/wuzhiyou/plumed/bin/plumed"  # 若已在 PATH 里可改 "plumed"

# 把 gmx 所在目录加进 PATH(这样 subprocess 里直接写 "gmx" 也能找到)
os.environ["PATH"] = GMX_BIN_DIR + os.pathsep + os.environ["PATH"]

# 只有当你的 GROMACS 不是内置 PLUMED、需要外部 kernel 时才需要这行。
# 见下方说明,大多数 -plumed 补丁版其实不需要。
os.environ["PLUMED_KERNEL"] = PLUMED_KERNEL_PATH
step = 10
deg_to_rad = np.pi / 180.0


# -------------------------------
# 0) Helper: convert discard fraction to frame count
# -------------------------------
def compute_discard_n(n_total, discard_frac):
    """
    Convert discard fraction to number of frames to discard.

    Parameters
    ----------
    n_total : int
        Total number of frames.
    discard_frac : float
        Fraction in [0, 1). Example: 0.2 means discard first 20%.

    Returns
    -------
    discard_n : int
        Number of frames to discard.
    """
    if discard_frac is None:
        return 0
    if not (0.0 <= discard_frac < 1.0):
        raise ValueError(
            f"discard_frac must satisfy 0 <= discard_frac < 1, got {discard_frac}"
        )
    discard_n = int(np.floor(n_total * discard_frac))
    if discard_n >= n_total and n_total > 0:
        raise ValueError(
            f"discard_frac={discard_frac} removes all frames: "
            f"n_total={n_total}, discard_n={discard_n}"
        )
    return discard_n


# -------------------------------
# 1) Run restrained simulation
# -------------------------------
def restrained_simulation(grid_point):
    phi_deg, psi_deg, phi, psi = grid_point
    dir_name = f"phi_{phi_deg}_psi_{psi_deg}"
    os.makedirs(dir_name, exist_ok=True)

    # Pre-equilibration (soft restraint)
    plumed_content_initial = f"""
MOLINFO STRUCTURE=./ala2.pdb

phi: TORSION ATOMS=@phi-2
psi: TORSION ATOMS=@psi-2
restraint: RESTRAINT ARG=phi,psi AT={phi},{psi} KAPPA=100.0,100.0

PRINT ARG=phi,psi,restraint.bias FILE={dir_name}/COLVAR_initial STRIDE=100
"""

    # Production (stiff restraint) + dump projections
    plumed_content = f"""
MOLINFO STRUCTURE=./ala2.pdb

phi: TORSION ATOMS=@phi-2
psi: TORSION ATOMS=@psi-2
restraint: RESTRAINT ARG=phi,psi AT={phi},{psi} KAPPA=1000.0,1000.0

DUMPPROJECTIONS ARG=phi,psi FILE={dir_name}/PROJ STRIDE=20
PRINT ARG=phi,psi,restraint.bias FILE={dir_name}/COLVAR_restrained STRIDE=20
"""

    with open(f"{dir_name}/plumed.dat", "w") as f:
        f.write(plumed_content)

    with open(f"{dir_name}/plumed_initial.dat", "w") as f:
        f.write(plumed_content_initial)

    # Run: short soft stage -> prepare new tpr -> stiff restrained stage
    subprocess.run(
        f"gmx mdrun -s nvt.tpr -plumed {dir_name}/plumed_initial.dat "
        f"-nb cpu -pme cpu -ntmpi 1 -deffnm {dir_name}/initial -v",
        shell=True,
        check=True,
    )

    subprocess.run(
        f"gmx grompp -f nvt.mdp -c {dir_name}/initial.gro -r {dir_name}/initial.gro "
        f"-p topol.top -o {dir_name}/restrained.tpr",
        shell=True,
        check=True,
    )

    subprocess.run(
        f"gmx mdrun -s {dir_name}/restrained.tpr -plumed {dir_name}/plumed.dat "
        f"-nb cpu -pme cpu -ntmpi 1 -deffnm {dir_name}/restrained -v",
        shell=True,
        check=True,
    )


# -------------------------------
# 2) Plot helper
# -------------------------------
def plot_colvar(filename, figname):
    colvar = np.loadtxt(filename, comments="#")
    if colvar.ndim == 1:
        colvar = colvar.reshape(1, -1)

    plt.figure(figsize=(10, 6))
    plt.scatter(
        colvar[:, 1] / deg_to_rad,
        colvar[:, 2] / deg_to_rad,
        c=colvar[:, 3],
        alpha=0.5,
    )
    plt.colorbar(label="Restraint Bias")
    plt.xlabel("phi (deg)")
    plt.ylabel("psi (deg)")
    plt.title("COLVAR Analysis")
    plt.savefig(figname)
    plt.close()


# -------------------------------
# 3) Parse DUMPPROJECTIONS output
#    Expected columns:
#    time, g11, g12, g21, g22
# -------------------------------
def metric_from_proj(proj_path, tmin=None, tmax=None, discard_frac=0.0):
    arr = np.loadtxt(proj_path, comments="#")
    if arr.ndim == 1:
        arr = arr.reshape(1, -1)

    t = arr[:, 0]
    g11, g12, g21, g22 = arr[:, 1:5].T

    # Optional time window
    mask = np.ones_like(t, dtype=bool)
    if tmin is not None:
        mask &= (t >= tmin)
    if tmax is not None:
        mask &= (t <= tmax)

    t = t[mask]
    g11 = g11[mask]
    g12 = g12[mask]
    g21 = g21[mask]
    g22 = g22[mask]

    n_total = len(t)
    if n_total == 0:
        raise ValueError(f"No frames left in {proj_path} after applying tmin/tmax filter.")

    discard_n = compute_discard_n(n_total, discard_frac)
    print(f"{proj_path}: discarding {discard_n}/{n_total} frames ({discard_frac:.1%})")

    if discard_n > 0:
        t = t[discard_n:]
        g11 = g11[discard_n:]
        g12 = g12[discard_n:]
        g21 = g21[discard_n:]
        g22 = g22[discard_n:]

    n = len(t)
    if n == 0:
        raise ValueError(f"No frames left in {proj_path} after discarding discard_frac={discard_frac}.")

    M11 = g11.mean()
    M22 = g22.mean()
    M12 = 0.5 * (g12.mean() + g21.mean())
    M = np.array([[M11, M12], [M12, M22]])

    def se(x):
        return (x.std(ddof=1) / np.sqrt(n)) if n > 1 else np.nan

    offdiag = 0.5 * (g12 + g21)
    M_se = np.array([
        [se(g11), se(offdiag)],
        [se(offdiag), se(g22)]
    ])

    return M, M_se, n


# -------------------------------
# 4) Free-energy gradient
# -------------------------------
def FES_gradient(grid_point, discard_frac=0.0, tmin=None, tmax=None, kappa=1000.0):
    phi_deg, psi_deg, phi, psi = grid_point
    dir_name = f"phi_{phi_deg}_psi_{psi_deg}"
    colvar_file = f"{dir_name}/COLVAR_restrained"

    colvar = np.loadtxt(colvar_file, comments="#")
    if colvar.ndim == 1:
        colvar = colvar.reshape(1, -1)

    t = colvar[:, 0]
    phi_restrained = colvar[:, 1]
    psi_restrained = colvar[:, 2]

    # Optional time window
    mask = np.ones_like(t, dtype=bool)
    if tmin is not None:
        mask &= (t >= tmin)
    if tmax is not None:
        mask &= (t <= tmax)

    t = t[mask]
    phi_restrained = phi_restrained[mask]
    psi_restrained = psi_restrained[mask]

    n_total = len(t)
    if n_total == 0:
        raise ValueError(f"No frames left in {colvar_file} after applying tmin/tmax filter.")

    discard_n = compute_discard_n(n_total, discard_frac)
    print(f"{colvar_file}: discarding {discard_n}/{n_total} frames ({discard_frac:.1%})")

    if discard_n > 0:
        t = t[discard_n:]
        phi_restrained = phi_restrained[discard_n:]
        psi_restrained = psi_restrained[discard_n:]

    if len(t) == 0:
        raise ValueError(f"No frames left in {colvar_file} after discarding discard_frac={discard_frac}.")

    dphi = phi_restrained - phi
    dpsi = psi_restrained - psi

    # Wrap to (-pi, pi]
    dphi = (dphi + np.pi) % (2 * np.pi) - np.pi
    dpsi = (dpsi + np.pi) % (2 * np.pi) - np.pi

    gradient_phi = -kappa * np.mean(dphi)
    gradient_psi = -kappa * np.mean(dpsi)

    return gradient_phi, gradient_psi


# -------------------------------
# 5) Metric assembly
# -------------------------------
def FES_metric(grid_point, discard_frac=0.0, tmin=None, tmax=None):
    phi_deg, psi_deg, phi, psi = grid_point
    dir_name = f"phi_{phi_deg}_psi_{psi_deg}"
    proj_file = os.path.join(dir_name, "PROJ")

    if not os.path.exists(proj_file):
        raise FileNotFoundError(
            f"Cannot find {proj_file}; make sure DUMPPROJECTIONS ran."
        )

    M, M_se, n = metric_from_proj(
        proj_file,
        tmin=tmin,
        tmax=tmax,
        discard_frac=discard_frac
    )
    return M


# -------------------------------
# 6) Main interface
# -------------------------------
def M_gradient(q, discard_frac=0.2, tmin=None, tmax=None):
    """
    Input:
        q: torch.Tensor, shape (1, 2), (phi, psi) in rad

    Parameters:
        discard_frac: float
            Fraction of initial frames to discard, e.g. 0.2 means discard 20%.

    Returns:
        M: np.ndarray, shape (2, 2)
        F_orig/20: np.ndarray, shape (2,)
    """
    q = q.cpu().detach().numpy()  # shape (1, 2)
    phi_deg_recovered = q[:, 0] / deg_to_rad
    psi_deg_recovered = q[:, 1] / deg_to_rad

    # g: [phi_deg, psi_deg, phi(rad), psi(rad)]
    g = np.column_stack((phi_deg_recovered, psi_deg_recovered, q))[0]

    # Run restrained simulation
    restrained_simulation(g)
    delete_backupfile()

    # Free-energy gradient
    grad_phi, grad_psi = FES_gradient(
        g,
        discard_frac=discard_frac,
        tmin=tmin,
        tmax=tmax,
        kappa=1000.0,
    )
    F_orig = np.array([grad_phi, grad_psi], dtype=float)

    # Metric
    M = FES_metric(
        g,
        discard_frac=discard_frac,
        tmin=tmin,
        tmax=tmax,
    )

    subprocess.run(
        'find . -type d -name "phi_*" -exec rm -rf {} +',
        shell=True,
    )

    return M, F_orig 


# -------------------------------
# 7) Housekeeping
# -------------------------------
def delete_backupfile():
    directory = "."
    backup_pattern = os.path.join(directory, "bck.*.PLUMED.OUT")
    for file_path in glob.glob(backup_pattern):
        try:
            os.remove(file_path)
            print(f"Deleted: {file_path}")
        except OSError as e:
            print(f"Error deleting {file_path}: {e}")
    print("Done deleting backup files.")
    
    
    
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
import torch
from mpl_toolkits.axes_grid1 import make_axes_locatable


def to_numpy(x):
    if x is None:
        return None
    if isinstance(x, torch.Tensor):
        x = x.detach().cpu().numpy()
    return np.asarray(x, dtype=float)


def crop_regular_grid(X, Y, Z, xlim=None, ylim=None):
    """
    Crop a regular 2D meshgrid X, Y, Z to the rectangle defined by xlim, ylim.
    """
    if xlim is None:
        xlim = (float(np.nanmin(X)), float(np.nanmax(X)))
    else:
        xlim = tuple(sorted(xlim))

    if ylim is None:
        ylim = (float(np.nanmin(Y)), float(np.nanmax(Y)))
    else:
        ylim = tuple(sorted(ylim))

    inside = (X >= xlim[0]) & (X <= xlim[1]) & (Y >= ylim[0]) & (Y <= ylim[1])

    if not np.any(inside):
        raise ValueError(
            f"No grid points inside xlim={xlim}, ylim={ylim}. "
            f"Available X range: [{np.nanmin(X):.6f}, {np.nanmax(X):.6f}], "
            f"Y range: [{np.nanmin(Y):.6f}, {np.nanmax(Y):.6f}]"
        )

    rows = np.where(np.any(inside, axis=1))[0]
    cols = np.where(np.any(inside, axis=0))[0]

    r0, r1 = rows[0], rows[-1] + 1
    c0, c1 = cols[0], cols[-1] + 1

    return X[r0:r1, c0:c1], Y[r0:r1, c0:c1], Z[r0:r1, c0:c1], xlim, ylim


def compute_figsize_from_window(xlim, ylim, scale_factor=1.0, base=6.0):
    dx = max(abs(xlim[1] - xlim[0]), 1e-12)
    dy = max(abs(ylim[1] - ylim[0]), 1e-12)
    base = base * scale_factor

    aspect = dx / dy
    if aspect >= 1:
        return (base, max(4.0, base / aspect))
    else:
        return (max(4.0, base * aspect), base)


def visualize_fes_zoomin_from_npz(
    k,
    x_plot1=None,
    x_plot2=None,
    x_plot3=None,
    x_plot4=None,
    force1=None,
    force2=None,
    force3=None,
    npz_path="/Users/wuzhiyou/Code/IVM/MFEP_ala/amber14sb_vacuo/wtmeta_phi_psi/fes_XYZ.npz",
    title="Potential Surface",
    xlim=(-np.pi, np.pi),
    ylim=(-np.pi, np.pi),
    scale_factor=1.0,
):
    # -----------------------------
    # Load X, Y, Z
    # -----------------------------
    data = np.load(npz_path)

    grid_x = np.asarray(data["X"], dtype=float)
    grid_y = np.asarray(data["Y"], dtype=float)
    grid_z = np.ma.masked_invalid(np.asarray(data["Z"], dtype=float))

    # If X/Y are 1D, convert to meshgrid
    if grid_x.ndim == 1 and grid_y.ndim == 1:
        grid_x, grid_y = np.meshgrid(grid_x, grid_y, indexing="xy")

    if grid_x.shape != grid_y.shape or grid_x.shape != grid_z.shape:
        raise ValueError(f"Shape mismatch: X{grid_x.shape}, Y{grid_y.shape}, Z{grid_z.shape}")

    # -----------------------------
    # Convert coordinate grid to radians
    # Assumption: npz stores X/Y in degrees
    # If your npz is already in radians, delete these two lines
    # -----------------------------
    grid_x = np.deg2rad(grid_x)
    grid_y = np.deg2rad(grid_y)

    # -----------------------------
    # Convert optional inputs to numpy
    # These are assumed to already be in radians
    # -----------------------------
    x_plot1 = to_numpy(x_plot1)
    x_plot2 = to_numpy(x_plot2)
    x_plot3 = to_numpy(x_plot3)
    x_plot4 = to_numpy(x_plot4)

    force1 = to_numpy(force1)
    force2 = to_numpy(force2)
    force3 = to_numpy(force3)

    # -----------------------------
    # Crop grid to requested radian window
    # -----------------------------
    X_plot, Y_plot, Z_plot, xlim_used, ylim_used = crop_regular_grid(
        grid_x, grid_y, grid_z, xlim=xlim, ylim=ylim
    )

    # -----------------------------
    # Figure and axis
    # -----------------------------
    figsize = compute_figsize_from_window(
        xlim_used, ylim_used, scale_factor=scale_factor
    )
    fig, ax = plt.subplots(figsize=figsize)

    z_min = float(np.ma.min(Z_plot))
    z_max = float(np.ma.max(Z_plot))

    if z_max > 40:
        print(z_min, 40, z_max)
        norm = mpl.colors.TwoSlopeNorm(vmin=z_min, vcenter=40, vmax=z_max)
        cp = ax.contourf(X_plot, Y_plot, Z_plot, levels=50, cmap=mpl.cm.jet, norm=norm)
    else:
        cp = ax.contourf(X_plot, Y_plot, Z_plot, levels=50, cmap=mpl.cm.jet)

    # -----------------------------
    # Plot trajectories
    # All trajectories are assumed to already be in radians
    # -----------------------------
    has_legend = False

    if x_plot1 is not None:
        ax.plot(
            x_plot1[:, 0], x_plot1[:, 1],
            color="green", linestyle="-", marker="o", markersize=4,
            label=r"x_hat_old"
        )
        has_legend = True

    if x_plot2 is not None:
        ax.plot(
            x_plot2[:, 0], x_plot2[:, 1],
            color="blue", linestyle="-", marker="o", markersize=4,
            label=r"x_*"
        )
        has_legend = True

    if x_plot3 is not None:
        ax.plot(
            x_plot3[:, 0], x_plot3[:, 1],
            color="purple", linestyle="-", marker="o", markersize=4,
            label=r"x_hat_old + P(x_* - x_hat_old)"
        )
        has_legend = True

    if x_plot4 is not None:
        ax.plot(
            x_plot4[:, 0], x_plot4[:, 1],
            color="red", linestyle="-", marker="o", markersize=4,
            label=r"x_reparam_new"
        )
        has_legend = True

    # -----------------------------
    # Plot force vectors
    # -----------------------------
    if x_plot1 is not None:
        traj_x = x_plot1[:, 0]
        traj_y = x_plot1[:, 1]

        base_quiver_kwargs = dict(
            angles="xy",
            scale_units="xy",
            scale=1,
        )

        if force1 is not None:
            if force1.shape != x_plot1.shape:
                raise ValueError(f"force1 shape {force1.shape} != x_plot1 shape {x_plot1.shape}")
            ax.quiver(
                traj_x, traj_y, force1[:, 0], force1[:, 1],
                color="purple",
                width=0.006,
                headwidth=4,
                headlength=5,
                **base_quiver_kwargs
            )

        if force2 is not None:
            if force2.shape != x_plot1.shape:
                raise ValueError(f"force2 shape {force2.shape} != x_plot1 shape {x_plot1.shape}")
            ax.quiver(
                traj_x, traj_y, force2[:, 0], force2[:, 1],
                color="orange",
                width=0.004,
                headwidth=4,
                headlength=5,
                **base_quiver_kwargs
            )

        if force3 is not None:
            if force3.shape != x_plot1.shape:
                raise ValueError(f"force3 shape {force3.shape} != x_plot1 shape {x_plot1.shape}")
            ax.quiver(
                traj_x, traj_y, force3[:, 0], force3[:, 1],
                color="pink",
                width=0.003,
                headwidth=4,
                headlength=5,
                **base_quiver_kwargs
            )

    # -----------------------------
    # Mark special points
    # -----------------------------
    special_points = [
        (-2.62593029,  2.77365591, r"$C5$"),
        (-1.33433281,  0.99268968, r"$C7_{eq}$"),
    ]

    for x, y, label in special_points:
        ax.plot(
            x, y,
            marker="x",
            color="white",
            markersize=10,
            markeredgewidth=2.5,
            linestyle="None",
            zorder=10,
        )
        ax.text(
            x - 0.5, y -0.5, label,
            color="white",
            fontsize=30,
            weight="bold",
            zorder=10,
        )

    # -----------------------------
    # Colorbar with same height as main plot
    # -----------------------------
    divider = make_axes_locatable(ax)
    cax = divider.append_axes("right", size="4.5%", pad=0.12)
    cbar = fig.colorbar(cp, cax=cax)
    cbar.set_label("Potential")

    # -----------------------------
    # Axis settings
    # -----------------------------
    ax.set_xlabel(r"$\phi$ (rad)")
    ax.set_ylabel(r"$\psi$ (rad)")
    # ax.set_title(f"{title}")

    ax.set_xlim(xlim_used)
    ax.set_ylim(ylim_used)
    ax.set_aspect("equal", adjustable="box")

    if has_legend:
        ax.legend()

    plt.tight_layout()
    fig.savefig("potential_surface.pdf", format="pdf", bbox_inches="tight")
    plt.show()

    return fig, X_plot, Y_plot, Z_plot


fig, X_plot, Y_plot, Z_plot = visualize_fes_zoomin_from_npz(
    k=0,
    title="Potential Surface",
    xlim=(-np.pi, np.pi),
    ylim=(-np.pi, np.pi),
    scale_factor=1.5,
)
def fig_contour(k, x_plot1, x_plot2, x_plot3, x_plot4, xlim, ylim, force1=None, force2=None, force3=None, scale_factor=1.3):
    """
    x_plot: [N, 2] points
    force:  [N, 2] force vectors evaluated at x_plot (optional)
    """
    visualize_fes_zoomin_from_npz(
        k,
        x_plot1=x_plot1,
        x_plot2=x_plot2,
        x_plot3=x_plot3,
        x_plot4=x_plot4,
        force1=force1,          # <--- pass force to show arrows
        force2=force2,
        force3=force3,
        xlim=xlim,
        ylim=ylim,
        scale_factor=scale_factor,
    )

def fig_save(k, x_plot1, x_plot2, x_plot3, x_plot4, xlim, ylim, force1=None, force2=None, force3=None, scale_factor=1.3, figure_path = None):
    """
    x_plot: [N, 2] points
    force:  [N, 2] force vectors evaluated at x_plot (optional)
    """
    fig, _, _, _ = visualize_fes_zoomin_from_npz(
        k,
        x_plot1=x_plot1,
        x_plot2=x_plot2,
        x_plot3=x_plot3,
        x_plot4=x_plot4,
        force1=force1,          # <--- pass force to show arrows
        force2=force2,
        force3=force3,
        xlim=xlim,
        ylim=ylim,
        scale_factor=scale_factor,
    )
    fig.savefig(os.path.join(figure_path, f"images_k{k}.png"), dpi=300)

In [ ]:
import numpy as np
import torch


def _grad_at(q_np, discard_frac=0.2, tmin=None, tmax=None):
    """在点 q=(phi, psi)[rad] 处调用一次 MD，返回自由能梯度(平均力)。"""
    qt = torch.tensor(np.asarray(q_np, dtype=np.float64).reshape(1, 2))
    _, F = M_gradient(qt, discard_frac=discard_frac, tmin=tmin, tmax=tmax)
    return np.asarray(F, dtype=float)


def _wrap(x):
    """把角度 wrap 到 (-pi, pi]。"""
    return (np.asarray(x) + np.pi) % (2 * np.pi) - np.pi


def find_minimum_bfgs(
    q0,
    discard_frac=0.2,
    tmin=None,
    tmax=None,
    max_iter=40,
    gtol=0.05,        # |grad| 收敛阈值 (kJ/mol/rad)，按噪声水平调
    h0=1e-3,          # 初始逆 Hessian 标度，控制第一步步长
    max_move=0.3,     # 单步最大位移 (rad)，防止乱跳
):
    """
    只用梯度的 BFGS，寻找自由能极小值。
    q0 : array-like, (2,)  初始 (phi, psi) [rad]
    返回: (q_min[2], grad_at_min[2])
    """
    q = np.array(q0, dtype=float).reshape(2)
    H = h0 * np.eye(2)          # 逆 Hessian 近似
    g = _grad_at(q, discard_frac, tmin, tmax)

    for it in range(max_iter):
        gnorm = np.linalg.norm(g)
        print(f"[BFGS] iter {it:2d}: q=({q[0]:+.4f},{q[1]:+.4f})  |g|={gnorm:.3f}")
        if gnorm < gtol:
            print(f"[BFGS] converged: |g|={gnorm:.3f} < gtol={gtol}")
            break

        # 拟牛顿搜索方向（步长隐含 alpha=1，由 H 的标度决定）
        p = -H @ g
        # 步长上限，防止在噪声梯度下飞出
        if np.linalg.norm(p) > max_move:
            p = p * (max_move / np.linalg.norm(p))

        q_new = _wrap(q + p)
        g_new = _grad_at(q_new, discard_frac, tmin, tmax)

        # BFGS 逆 Hessian 更新（用真实周期位移 s）
        s = _wrap(q_new - q)
        y = g_new - g
        sy = float(s @ y)
        if sy > 1e-10:                       # 曲率条件，保证正定
            rho = 1.0 / sy
            I = np.eye(2)
            H = (I - rho * np.outer(s, y)) @ H @ (I - rho * np.outer(y, s)) \
                + rho * np.outer(s, s)

        q, g = q_new, g_new

    return q, g


# -------------------------------
# 用 BFGS 找两个端点，替换原来手动给定的端点
# -------------------------------
q_start_A = np.array([-1.3304716, 0.99118795])   # 出发点 A (rad)
q_start_B = np.array([1.07356198, -0.7456166])   # 出发点 B (rad)#1.07290893 -0.7363371

min_A, gA = find_minimum_bfgs(q_start_A, discard_frac=0.2)
min_B, gB = find_minimum_bfgs(q_start_B, discard_frac=0.2)

print(f"Endpoint A (from (-3, 3)): {min_A}  |g|={np.linalg.norm(gA):.3f}  "
      f"deg=({min_A[0]/deg_to_rad:.1f}, {min_A[1]/deg_to_rad:.1f})")
print(f"Endpoint B (from (-1, 1)): {min_B}  |g|={np.linalg.norm(gB):.3f}  "
      f"deg=({min_B[0]/deg_to_rad:.1f}, {min_B[1]/deg_to_rad:.1f})")

# 把找到的极小值作为端点替换进去
endpoint1 = min_A
endpoint2 = min_B

In [ ]:
-

In [ ]:
# import os
# import numpy as np
# import matplotlib.pyplot as plt
# import torch
# import torch.nn as nn
# import time


# # ============================================================
# # 外部依赖（来自你的第一个脚本，保持不变）:
# #   M_gradient(q, discard_frac, tmin, tmax) -> (M (2,2), F_orig (2,))
# #   fig_contour(...), fig_save(...), deg_to_rad
# # ============================================================

# device = torch.device("cpu")
# dtype = torch.float64
# N_dim = 2


# # ------------------------------------------------------------
# # 1) FMEP 梯度/度量封装 —— 度量部分完全沿用你的版本
# #    返回: Mg = M∇F (rescaled, 仅诊断用), M_seq = [M_i], g = ∇F
# # ------------------------------------------------------------
# def gradient_md_use(q, discard_frac=0.05, tmin=None, tmax=None):
#     t0 = time.perf_counter()
#     if not torch.is_tensor(q):
#         raise TypeError(f"q must be a torch.Tensor, got {type(q)}")
#     if q.ndim == 1:
#         q = q.unsqueeze(0)
#     if q.ndim != 2 or q.shape[1] != 2:
#         raise ValueError(f"q must have shape (N, 2), got {tuple(q.shape)}")

#     out_device = q.device
#     out_dtype = torch.float64
#     q_cpu = q.detach().cpu()
#     n_points = q_cpu.shape[0]

#     def _to_numpy(x):
#         if torch.is_tensor(x):
#             x = x.detach().cpu().numpy()
#         return np.asarray(x, dtype=float)

#     Mg_list, M_seq, g_list = [], [], []
#     for idx in range(n_points):
#         q_i = q_cpu[idx:idx + 1]
#         M_i, F_i = M_gradient(q_i, discard_frac=discard_frac, tmin=tmin, tmax=tmax)
#         M_i = _to_numpy(M_i).reshape(2, 2)
#         F_i = _to_numpy(F_i).reshape(2,)
#         F_i = F_i / 4.184                     # kJ/mol -> kcal/mol
#         Mg_i = M_i @ F_i
#         Mg_norm = np.linalg.norm(Mg_i)
#         f_norm = np.linalg.norm(F_i)
#         # if Mg_norm > 1e-14:
#         #     Mg_i = Mg_i * (f_norm / Mg_norm)  # rescale (仅影响诊断量, 不影响零点)
#         Mg_list.append(Mg_i)
#         M_seq.append(torch.tensor(M_i, dtype=out_dtype, device=out_device))
#         g_list.append(F_i)

#     Mg = torch.tensor(np.stack(Mg_list, 0), dtype=out_dtype, device=out_device)
#     g = torch.tensor(np.stack(g_list, 0), dtype=out_dtype, device=out_device)
#     t1 = time.perf_counter()
#     print(f"serial wall time = {t1 - t0:.3f} s  ({n_points} points)")
#     return Mg, M_seq, g


# # ------------------------------------------------------------
# # 2) Image (node) 路径表示
# # ------------------------------------------------------------
# class PathPoints(nn.Module):
#     def __init__(self, x0, x1, N, custom_init=None):
#         super().__init__()
#         x0 = torch.as_tensor(x0, dtype=torch.float64)
#         x1 = torch.as_tensor(x1, dtype=torch.float64)
#         self.d = x0.numel()
#         self.N = N
#         self.register_buffer("x0", x0)
#         self.register_buffer("x1", x1)

#         if custom_init is not None:
#             P_init = torch.as_tensor(custom_init, dtype=torch.float64)
#         else:
#             t = torch.linspace(0.0, 1.0, N + 2, dtype=torch.float64)[1:-1]
#             P_init = (1 - t).unsqueeze(1) * x0 + t.unsqueeze(1) * x1

#         self.P_list = nn.ParameterList(
#             [nn.Parameter(P_init[i].unsqueeze(0)) for i in range(N)]
#         )

#     def get_full_path(self):
#         P = torch.cat([p for p in self.P_list], dim=0)
#         return torch.vstack([self.x0.unsqueeze(0), P, self.x1.unsqueeze(0)])

#     def forward(self):
#         return torch.cat([p for p in self.P_list], dim=0)


# @torch.no_grad()
# def set_model_internal_positions(model, X_NxD):
#     X_NxD = X_NxD.detach().to(
#         device=model.P_list[0].device, dtype=model.P_list[0].dtype
#     )
#     for j, p in enumerate(model.P_list):
#         p.data = X_NxD[j:j + 1].clone()


# def central_difference_tangent(X_full):
#     # X_full: (N+2, d) -> 内部 image 的中心差分切向 (N, d)
#     return X_full[2:] - X_full[:-2]


# # ------------------------------------------------------------
# # 3) G-正交投影算子 —— 沿用你的版本
# # ------------------------------------------------------------
# def G_projectors(partial_s_x, M_seq):
#     N, d = partial_s_x.shape
#     assert d == 2
#     dev = partial_s_x.device
#     dt = partial_s_x.dtype

#     G_list, tG_list = [], []
#     for i in range(N):
#         M_i = M_seq[i]
#         G_i = torch.linalg.inv(M_i)               # G = M^{-1}
#         v_i = partial_s_x[i:i + 1, :]
#         Gv = v_i @ G_i
#         norm2 = torch.sum(v_i * Gv, dim=1, keepdim=True) + 1e-16
#         norm_G = torch.sqrt(norm2)
#         tG_i = v_i / norm_G
#         G_list.append(G_i)
#         tG_list.append(tG_i)

#     G_all = torch.stack(G_list, dim=0)            # (N,2,2)
#     tG_all = torch.cat(tG_list, dim=0)            # (N,2)

#     tG_all_3d = tG_all.unsqueeze(2)               # (N,2,1)
#     G_tG = torch.bmm(G_all, tG_all_3d)            # (N,2,1)
#     P_parallel_mats = torch.bmm(tG_all_3d, G_tG.transpose(1, 2))
#     denom = torch.sum(tG_all * G_tG.squeeze(-1), dim=1, keepdim=True)
#     denom = denom.view(-1, 1, 1) + 1e-16
#     P_parallel_mats = P_parallel_mats / denom     # (N,2,2)

#     I = torch.eye(2, device=dev, dtype=dt).unsqueeze(0).expand(N, -1, -1)
#     P_perp_mats = I - P_parallel_mats             # (N,2,2)

#     def P_parallel_G(x):
#         return torch.bmm(P_parallel_mats, x.unsqueeze(-1)).squeeze(-1)

#     def P_perp_G(x):
#         return torch.bmm(P_perp_mats, x.unsqueeze(-1)).squeeze(-1)

#     return tG_all, P_parallel_G, P_perp_G, P_parallel_mats, P_perp_mats


# def P_perp_euclid(x, partial_s_x, eps=1e-16):
#     n = torch.sum(partial_s_x ** 2, axis=1, keepdim=True) ** 0.5
#     t = partial_s_x / (n + eps)
#     inner = torch.sum(t * x, axis=1, keepdim=True)
#     return x - inner * t


# # ------------------------------------------------------------
# # 4) NEB 切向弹簧重参数化 (论文 Algorithm 2)
# #    F_sp^i = k_s (l+ - l-) tau_hat_i ;  前向欧拉
# #    停机: max_i || F_sp^i || < tol   (k_s=1 时即 max|l+-l-|<tol)
# # ------------------------------------------------------------
# def reparam_by_neb(X_full, k_s=1.0, dtau=0.5, max_iter=5000,
#                    tol=1e-3, verbose=False):
#     X = X_full.clone().detach()
#     eps = 1e-16
#     resid = float("inf")
#     for r in range(max_iter):
#         seg = X[1:] - X[:-1]                                   # (N+1, d)
#         seg_len = torch.linalg.norm(seg, dim=1)                # (N+1,)
#         l_minus = seg_len[:-1]                                 # (N,)
#         l_plus = seg_len[1:]                                   # (N,)

#         tang = X[2:] - X[:-2]                                  # (N, d)
#         tang = tang / (torch.linalg.norm(tang, dim=1, keepdim=True) + eps)

#         F_sp = k_s * (l_plus - l_minus).unsqueeze(1) * tang    # (N, d)
#         resid = torch.max(torch.linalg.norm(F_sp, dim=1)).item()

#         X[1:-1] = X[1:-1] + dtau * F_sp                        # forward Euler

#         if resid < tol:
#             if verbose:
#                 print(f"  [reparam_by_neb] converged @ iter {r}, "
#                       f"max|F_sp|={resid:.3e}")
#             break
#     else:
#         if verbose:
#             print(f"  [reparam_by_neb] hit max_iter={max_iter}, "
#                   f"max|F_sp|={resid:.3e}")
#     return X, resid


# # ------------------------------------------------------------
# # 5) 沿路径线积分重建自由能剖面 (仅用于诊断绘图)
# # ------------------------------------------------------------
# def recover_V_along_curve(s, g, phi_prime, V_ref=0.0):
#     dVds = (g * phi_prime).sum(dim=1, keepdim=True)
#     ds = s[1:] - s[:-1]
#     inc = 0.5 * (dVds[:-1] + dVds[1:]) * ds
#     V = torch.zeros_like(dVds)
#     V[0] = V_ref
#     V[1:] = V_ref + torch.cumsum(inc, dim=0)
#     return V - (torch.min(V) - 1e-6), dVds


# # ------------------------------------------------------------
# # 6) 绘图辅助 —— 沿用你的定义
# # ------------------------------------------------------------
# def fig_cos_V_force(i, s, arc, cos, g, v, x_pred, norm_F_perp,
#                     folder_path, mycase, casenum):
#     force = np.sqrt(np.sum(g * g, axis=1))
#     sin_square = 1 - cos ** 2
#     fig, axs = plt.subplots(1, 5, figsize=(20, 4))
#     axs[0].plot(arc, cos, 'b.'); axs[0].set_ylim(-1.1, 1.1); axs[0].set_title("cos"); axs[0].set_box_aspect(1)
#     axs[1].plot(arc, sin_square, 'b.'); axs[1].set_ylim(-0.1, 1.1); axs[1].set_title("1-cos^2"); axs[1].set_box_aspect(1)
#     axs[2].plot(arc.flatten(), v, 'b.'); axs[2].set_title("Free energy"); axs[2].set_box_aspect(1)
#     axs[3].plot(arc, force, 'b.'); axs[3].set_title("Force"); axs[3].set_box_aspect(1)
#     axs[4].plot(arc, norm_F_perp, 'b.'); axs[4].set_title("norm F_perp"); axs[4].set_box_aspect(1)
#     plt.tight_layout()
#     if i % 50 == 0 or i in [1, 100, 200]:
#         plt.savefig(os.path.join(folder_path, f"Dot_plot_Iter{i}_{mycase}_{casenum}.eps"))
#     plt.show()


# def fig_loss_batch(loss, loss_reparam, loss_1, loss_IVP, loss_3, loss_EL):
#     fig, axes = plt.subplots(1, 5, figsize=(25, 4))
#     axes[0].semilogy(loss, color="#B41830", label="I_perp")
#     axes[0].semilogy(loss_reparam, color="#4F18B4", label="reparam resid")
#     axes[0].set_title("I_perp & reparam"); axes[0].legend()
#     axes[1].semilogy(loss_1, color="#B41830"); axes[1].set_title("fmax")
#     axes[2].semilogy(loss_IVP, color="#B41830"); axes[2].set_title("I_perp")
#     axes[3].semilogy(loss_3, color="#B41830"); axes[3].set_title("1-cos^2")
#     axes[4].semilogy(loss_EL, color="#B41830"); axes[4].set_title("fmax (EL)")
#     plt.tight_layout(); plt.show()


# # ============================================================
# # 全局固定设置 (不参与调参)
# # ============================================================
# C1 = [-2.60866606, 2.77661136]
# C2 = [-1.3304716, 0.99118795]
# N = 20
# seed = 41
# xlim = (-np.pi, 0)
# ylim = (0, np.pi)

# REEVALUATE_GRAD_INNER = True   # True: 每个内步在 x_eval 重算 ∇F
# REPARAM_KS = 1.0
# REPARAM_DTAU = 1e-3
# REPARAM_TOL = 1e-3
# REPARAM_MAXITER = 5000
# REPARAM_EVERY = 1

# base_path = '/Users/wuzhiyou/Code/IVM/MFEP_ala/amber14sb_vacuo'

# # 调参时关掉画图 (i%1==0 恒真, 否则每步弹 4 组图会卡死内核)
# # 想恢复单次运行的可视化, 设为 True
# SHOW_FIGS = True


# # ============================================================
# # 单个配置的完整训练流程 (主循环逐字沿用你的版本, 只包了一层函数)
# # ============================================================
# def run_experiment(N_sub, alpha_IVP, eta_in):
#     # 约束: N_sub * batches = 300
#     batches = 200 // N_sub                     # 4->75, 5->60, 10->30

#     # 复位随机种子, 保证每个配置从相同初始路径出发
#     np.random.seed(seed)
#     torch.manual_seed(seed)

#     mycase = "alanine"
#     casenum = f"MFEP_image_Adam_N{N}_Nsub{N_sub}_alpha{alpha_IVP}_lr{eta_in}_NEBreparam"
#     folder_path = os.path.join(base_path, casenum)
#     os.makedirs(folder_path, exist_ok=True)
#     figure_path = os.path.join(folder_path, "figures")
#     os.makedirs(figure_path, exist_ok=True)

#     print("=" * 78)
#     print(f"[GRID] N_sub={N_sub}  alpha_IVP={alpha_IVP}  eta_in={eta_in}  "
#           f"batches={batches}  (N_sub*batches={N_sub*batches})")
#     print("=" * 78)

#     # 初始路径: 直线插值
#     model = PathPoints(C1, C2, N).to(device).double()

#     # 诊断序列
#     loss_perp_seq = []
#     loss_reparam_seq = []
#     loss_fmax_seq = []
#     loss_cos_seq = []
#     index_seq = []
#     data = {}

#     # --------------------------------------------------------
#     # 主循环: image + Adam 内层 + NEB 重参数化 (无回退)
#     # --------------------------------------------------------
#     for i in range(batches + 1):
#         # ----------------------------------------------------
#         # 冻结当前参考路径 φ_k = x_hat 的几何量
#         # ----------------------------------------------------
#         X_full = model.get_full_path().detach()          # (N+2, 2)
#         x_hat = model().detach()                          # (N, 2)
#         tangent = central_difference_tangent(X_full)      # (N, 2) 中心差分切向

#         # M(φ_k) 与 ∇F(φ_k)
#         Mg_ref, M_seq, g_ref = gradient_md_use(x_hat)
#         G_all = torch.stack([torch.linalg.inv(M_i) for M_i in M_seq], dim=0)  # (N,2,2)

#         # 冻结的 G-正交投影
#         (tG_hat, P_par_G, P_perp_G,
#          P_par_mats, P_perp_mats) = G_projectors(tangent, M_seq)

#         M_all = torch.stack(M_seq)        # (N,2,2) 冻结度量
#         RESCALE_MG = False

#         def build_perpG_Mg(g_now):
#             grad_G_F = torch.bmm(M_all, g_now.unsqueeze(-1)).squeeze(-1)
#             if RESCALE_MG:
#                 gn = torch.linalg.norm(g_now, dim=1, keepdim=True)
#                 mn = torch.linalg.norm(grad_G_F, dim=1, keepdim=True)
#                 grad_G_F = grad_G_F * (gn / (mn + 1e-14))
#             return torch.bmm(P_perp_mats, grad_G_F.unsqueeze(-1)).squeeze(-1)

#         # ----------------------------------------------------
#         # 诊断量 (参考路径处)
#         # ----------------------------------------------------
#         perpG_Mg = P_perp_G(Mg_ref)                       # P^perp grad_G F
#         norm_F_perp = torch.sum(perpG_Mg ** 2, axis=1, keepdim=True) ** 0.5
#         fmax = float(np.max(norm_F_perp.detach().cpu().numpy()))

#         tang_e = tangent / (torch.sum(tangent ** 2, axis=1, keepdim=True) ** 0.5 + 1e-16)
#         Mg_perp = P_perp_euclid(Mg_ref, tang_e)

#         # G-度量下的对齐角: <φ',grad_G F>_G = φ'^T ∇F
#         inner_G = torch.sum(tangent * g_ref, axis=1, keepdim=True)
#         tGt = torch.sum(tangent * torch.bmm(G_all, tangent.unsqueeze(-1)).squeeze(-1),
#                         axis=1, keepdim=True) ** 0.5
#         gMg = torch.sum(g_ref * torch.bmm(torch.stack(M_seq), g_ref.unsqueeze(-1)).squeeze(-1),
#                         axis=1, keepdim=True) ** 0.5
#         cos = inner_G / (tGt * gMg + 1e-16)
#         cos2 = cos ** 2

#         # I_perp = mean( |P^perp grad_G F|_G^2 * w )   (w 取单位权简化诊断)
#         I_perp = torch.mean(norm_F_perp ** 2).item()

#         # 自由能剖面 (线积分, 仅绘图)
#         s_col = torch.linspace(0, 1, N, dtype=dtype).unsqueeze(1)
#         ds = 1.0 / (N - 1)
#         phi_prime = torch.zeros_like(x_hat)
#         phi_prime[1:-1] = (x_hat[2:] - x_hat[:-2]) / (2 * ds)
#         phi_prime[0] = (x_hat[1] - x_hat[0]) / ds
#         phi_prime[-1] = (x_hat[-1] - x_hat[-2]) / ds
#         V_vals, _ = recover_V_along_curve(s_col, g_ref, phi_prime, V_ref=0.0)
#         v = V_vals.cpu().detach().numpy().flatten()

#         # ----------------------------------------------------
#         # 内层 Adam 求解 (论文 Algorithm 1, 式 32)
#         # ----------------------------------------------------
#         delta = nn.Parameter(torch.zeros(N, 2, dtype=dtype, device=device))
#         opt_inner = torch.optim.Adam([delta], lr=eta_in)

#         perpG_Mg_ref = None   # δ=0 (未形变) 时的准确 (M∇F)^perp

#         for i_sub in range(N_sub):
#             with torch.no_grad():
#                 current_x = torch.bmm(P_perp_mats, delta.detach().unsqueeze(-1)).squeeze(-1)
#                 x_eval = x_hat + current_x

#             if i_sub == 0 or (not REEVALUATE_GRAD_INNER):
#                 g = g_ref
#             else:
#                 _, _, g = gradient_md_use(x_eval)

#             with torch.no_grad():
#                 perpG_Mg_eval = build_perpG_Mg(g)
#                 if i_sub == 0:
#                     # δ=0: 尚未形变, ∇F 与 (M∇F)^perp 都是"准"的, 存档
#                     perpG_Mg_ref = perpG_Mg_eval.clone()
#                 dI_ddelta = perpG_Mg_eval + (1.0 / alpha_IVP) * current_x

#             opt_inner.zero_grad()
#             delta.grad = dI_ddelta.detach().clone()
#             opt_inner.step()

#         # ----------------------------------------------------
#         # 提交: φ_{k+1} = φ_k + P^perp δ*   (法向更新, 无回退)
#         # ----------------------------------------------------
#         with torch.no_grad():
#             current_x = torch.bmm(P_perp_mats, delta.detach().unsqueeze(-1)).squeeze(-1)
#             x_new = x_hat + current_x

#         max_step = float(torch.max(torch.linalg.norm(current_x, dim=1)).item())
#         print(f"[outer {i}] I_perp={I_perp:.4e}  fmax={fmax:.4e}  "
#               f"max_normal_step={max_step:.4e} rad")

#         set_model_internal_positions(model, x_new)

#         # ----------------------------------------------------
#         # NEB 切向弹簧重参数化
#         # ----------------------------------------------------
#         reparam_resid = 0.0
#         if REPARAM_EVERY > 0 and (i % REPARAM_EVERY == 0):
#             X_new_full = model.get_full_path().detach()
#             X_reparam, reparam_resid = reparam_by_neb(
#                 X_new_full, k_s=REPARAM_KS, dtau=REPARAM_DTAU,
#                 max_iter=REPARAM_MAXITER, tol=REPARAM_TOL, verbose=(i % 10 == 0)
#             )
#             set_model_internal_positions(model, X_reparam[1:-1])

#         x_reparam_new = model().detach()

#         # ----------------------------------------------------
#         # 记录 & (可选)绘图 & 保存
#         # ----------------------------------------------------
#         index_seq.append(i)
#         loss_perp_seq.append(I_perp)
#         loss_reparam_seq.append(reparam_resid if reparam_resid > 0 else 1e-16)
#         loss_fmax_seq.append(fmax)
#         loss_cos_seq.append(float(torch.mean(1 - cos2).item()))

#         if SHOW_FIGS and (i % 1 == 0 or i in [1, 50, 100, 200]):
#             fig_contour(i, x_hat.cpu().numpy(),
#                         x_reparam_new.cpu().numpy(), None, None,
#                         xlim=xlim, ylim=ylim)
#             fig_save(
#                 i * N_sub,
#                 x_hat.cpu().numpy(),
#                 x_new.cpu().numpy(),
#                 x_new.cpu().numpy(),
#                 x_reparam_new.cpu().numpy(),
#                 xlim=xlim, ylim=ylim,
#                 force1=-perpG_Mg.cpu().numpy(),
#                 force2=-g_ref.cpu().numpy(),
#                 force3=-Mg_perp.cpu().numpy(),
#                 figure_path=figure_path,
#             )
#             # fig_cos_V_force(
#             #     i, s_col.cpu().numpy(), s_col.cpu().numpy(),
#             #     cos.cpu().numpy(), g_ref.cpu().numpy(), v,
#             #     x_hat, norm_F_perp.cpu().numpy(),
#             #     folder_path, mycase, casenum
#             # )
#             # fig_loss_batch(
#             #     np.array(loss_perp_seq), np.array(loss_reparam_seq),
#             #     np.array(loss_fmax_seq), np.array(loss_perp_seq),
#             #     np.array(loss_cos_seq), np.array(loss_fmax_seq),
#             # )
#             torch.save(model.state_dict(),
#                        os.path.join(folder_path, f"model_MFEP_{mycase}_{casenum}_{i}.pth"))
#             plt.close('all')

#         data["index_seq"] = np.asarray(index_seq)
#         data["I_perp_seq"] = np.asarray(loss_perp_seq)
#         data["fmax_seq"] = np.asarray(loss_fmax_seq)
#         data["reparam_resid_seq"] = np.asarray(loss_reparam_seq)
#         data["cos_seq"] = np.asarray(loss_cos_seq)
#         data[f"path_iter_{i}"] = x_reparam_new.cpu().numpy()

#         # ---- δ=0 (未形变) 时每个 image 的准确量, 逐外迭代存档 ----
#         data[f"phi_ref_iter_{i}"]       = x_hat.cpu().numpy()            # (N,2) 参考路径 image
#         data[f"g_ref_iter_{i}"]         = g_ref.cpu().numpy()            # (N,2) ∇F @ δ=0
#         data[f"perpG_Mg_iter_{i}"]      = perpG_Mg_ref.cpu().numpy()     # (N,2) (M∇F)^perp @ δ=0
#         data[f"norm_perpG_Mg_iter_{i}"] = torch.linalg.norm(
#             perpG_Mg_ref, dim=1).cpu().numpy()                          # (N,) 每 image 法向力模

#         np.savez(os.path.join(folder_path, f"{mycase}_{casenum}.npz"), **data)

#     # 每个配置结束: 存最终 model + 返回最后一步诊断, 便于挑选最优参数
#     torch.save(model.state_dict(),
#                os.path.join(folder_path, f"model_MFEP_{mycase}_{casenum}_final.pth"))
#     return {
#         "N_sub": N_sub, "alpha_IVP": alpha_IVP, "eta_in": eta_in, "batches": batches,
#         "final_I_perp": loss_perp_seq[-1],
#         "final_fmax": loss_fmax_seq[-1],
#         "final_cos": loss_cos_seq[-1],
#         "min_I_perp": float(np.min(loss_perp_seq)),
#         "min_fmax": float(np.min(loss_fmax_seq)),
#     }


# # ============================================================
# # 断点续跑辅助函数
# #   done 判定: 优先看 _final.pth (只在整段跑完后才写),
# #              兜底看 npz 的 index_seq 是否已到最后一个外迭代
# # ============================================================
# def config_paths(N_sub, alpha_IVP, eta_in):
#     mycase = "alanine"
#     casenum = f"MFEP_image_Adam_N{N}_Nsub{N_sub}_alpha{alpha_IVP}_lr{eta_in}_NEBreparam"
#     folder_path = os.path.join(base_path, casenum)
#     final_model = os.path.join(folder_path, f"model_MFEP_{mycase}_{casenum}_final.pth")
#     npz_file = os.path.join(folder_path, f"{mycase}_{casenum}.npz")
#     return mycase, casenum, folder_path, final_model, npz_file


# def is_config_done(N_sub, alpha_IVP, eta_in):
#     batches = 300 // N_sub
#     _, _, _, final_model, npz_file = config_paths(N_sub, alpha_IVP, eta_in)
#     # 1) 最可靠标记: 收尾的 _final.pth 存在 => 整段跑完
#     if os.path.exists(final_model):
#         return True
#     # 2) 兜底: npz 里 index_seq 已到达最后一个外迭代 i == batches
#     if os.path.exists(npz_file):
#         try:
#             with np.load(npz_file, allow_pickle=True) as d:
#                 idx = d["index_seq"]
#                 if idx.size > 0 and int(np.max(idx)) >= batches:
#                     return True
#         except Exception:
#             return False
#     return False


# def load_result_from_npz(N_sub, alpha_IVP, eta_in):
#     """从已完成配置的 npz 里恢复诊断量, 让最终汇总表包含它。"""
#     batches = 300 // N_sub
#     _, _, _, _, npz_file = config_paths(N_sub, alpha_IVP, eta_in)
#     with np.load(npz_file, allow_pickle=True) as d:
#         I_perp_seq = np.asarray(d["I_perp_seq"], dtype=float)
#         fmax_seq = np.asarray(d["fmax_seq"], dtype=float)
#         cos_seq = np.asarray(d["cos_seq"], dtype=float)
#     return {
#         "N_sub": N_sub, "alpha_IVP": alpha_IVP, "eta_in": eta_in, "batches": batches,
#         "final_I_perp": float(I_perp_seq[-1]),
#         "final_fmax": float(fmax_seq[-1]),
#         "final_cos": float(cos_seq[-1]),
#         "min_I_perp": float(np.min(I_perp_seq)),
#         "min_fmax": float(np.min(fmax_seq)),
#     }


# # ============================================================
# # 网格搜索: N_sub × alpha_IVP × eta_in  (3×3×4 = 36 个配置)
# #   自动跳过已完成的配置, 只跑未完成/中断的
# # ============================================================
# GRID_N_SUB = [4, 5, 10]
# GRID_ALPHA = [0.1, 1, 10]
# GRID_ETA   = [0.03, 0.01, 0.003, 0.001]

# FORCE_RERUN = False   # 想强制全部重跑 (忽略已完成) 就设 True

# # --- 先扫描一遍, 报告进度 ---
# all_configs = [(ns, al, et)
#                for ns in GRID_N_SUB
#                for al in GRID_ALPHA
#                for et in GRID_ETA]
# done_flags = {cfg: is_config_done(*cfg) for cfg in all_configs}
# n_done = sum(done_flags.values())
# print("=" * 78)
# print(f"[RESUME] 共 {len(all_configs)} 个配置; 检测到已完成 {n_done} 个, "
#       f"待跑 {len(all_configs) - n_done} 个"
#       + ("  (FORCE_RERUN=True, 将全部重跑)" if FORCE_RERUN else ""))
# for cfg in all_configs:
#     ns, al, et = cfg
#     tag = "DONE  跳过" if (done_flags[cfg] and not FORCE_RERUN) else "TODO  待跑"
#     print(f"    [{tag}]  N_sub={ns:>2}  alpha={al:>4}  eta={et}")
# print("=" * 78)

# # --- 正式执行: 跳过已完成, 只跑未完成 ---
# summary = []
# for cfg in all_configs:
#     N_sub, alpha_IVP, eta_in = cfg
#     if done_flags[cfg] and not FORCE_RERUN:
#         print(f"[SKIP] N_sub={N_sub} alpha={alpha_IVP} eta={eta_in} 已完成, 直接读取旧结果")
#         try:
#             summary.append(load_result_from_npz(N_sub, alpha_IVP, eta_in))
#         except Exception as e:
#             print(f"       (读取旧结果失败, 汇总里将缺此项: {e})")
#         continue
#     res = run_experiment(N_sub, alpha_IVP, eta_in)
#     summary.append(res)


# # ---- 汇总所有配置, 存一份总表方便挑最优参数 ----
# print("\n" + "=" * 78)
# print("GRID SEARCH SUMMARY  (按 final_fmax 升序)")
# print("=" * 78)
# summary_sorted = sorted(summary, key=lambda r: r["final_fmax"])
# for r in summary_sorted:
#     print(f"N_sub={r['N_sub']:>2}  alpha={r['alpha_IVP']:>4}  eta={r['eta_in']:<6}  "
#           f"| final_I_perp={r['final_I_perp']:.3e}  final_fmax={r['final_fmax']:.3e}  "
#           f"final_1-cos2={r['final_cos']:.3e}  min_fmax={r['min_fmax']:.3e}")

# np.savez(os.path.join(base_path, "GRID_SUMMARY.npz"),
#          summary=np.array(summary, dtype=object))

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import time


# ============================================================
# 外部依赖（来自你的第一个脚本，保持不变）:
#   M_gradient(q, discard_frac, tmin, tmax) -> (M (2,2), F_orig (2,))
#   fig_contour(...), fig_save(...), deg_to_rad
# ============================================================

device = torch.device("cpu")
dtype = torch.float64
N_dim = 2


# ------------------------------------------------------------
# 1) FMEP 梯度/度量封装 —— 度量部分完全沿用你的版本
#    返回:
#       Mg = M∇F
#       M_seq = [M_i]
#       g = ∇F
# ------------------------------------------------------------
def gradient_md_use(q, discard_frac=0.05, tmin=None, tmax=None):
    t0 = time.perf_counter()

    if not torch.is_tensor(q):
        raise TypeError(f"q must be a torch.Tensor, got {type(q)}")

    if q.ndim == 1:
        q = q.unsqueeze(0)

    if q.ndim != 2 or q.shape[1] != 2:
        raise ValueError(f"q must have shape (N, 2), got {tuple(q.shape)}")

    out_device = q.device
    out_dtype = torch.float64
    q_cpu = q.detach().cpu()
    n_points = q_cpu.shape[0]

    def _to_numpy(x):
        if torch.is_tensor(x):
            x = x.detach().cpu().numpy()
        return np.asarray(x, dtype=float)

    Mg_list = []
    M_seq = []
    g_list = []

    for idx in range(n_points):
        q_i = q_cpu[idx:idx + 1]

        M_i, F_i = M_gradient(
            q_i,
            discard_frac=discard_frac,
            tmin=tmin,
            tmax=tmax,
        )

        M_i = _to_numpy(M_i).reshape(2, 2)
        F_i = _to_numpy(F_i).reshape(2,)

        # kJ/mol -> kcal/mol
        F_i = F_i / 4.184

        Mg_i = M_i @ F_i

        Mg_norm = np.linalg.norm(Mg_i)
        f_norm = np.linalg.norm(F_i)

        # if Mg_norm > 1e-14:
        #     Mg_i = Mg_i * (f_norm / Mg_norm)

        Mg_list.append(Mg_i)

        M_seq.append(
            torch.tensor(
                M_i,
                dtype=out_dtype,
                device=out_device,
            )
        )

        g_list.append(F_i)

    Mg = torch.tensor(
        np.stack(Mg_list, axis=0),
        dtype=out_dtype,
        device=out_device,
    )

    g = torch.tensor(
        np.stack(g_list, axis=0),
        dtype=out_dtype,
        device=out_device,
    )

    t1 = time.perf_counter()

    print(
        f"serial wall time = {t1 - t0:.3f} s  "
        f"({n_points} points)"
    )

    return Mg, M_seq, g


# ------------------------------------------------------------
# 2) Image (node) 路径表示
# ------------------------------------------------------------
class PathPoints(nn.Module):
    def __init__(self, x0, x1, N, custom_init=None):
        super().__init__()

        x0 = torch.as_tensor(x0, dtype=torch.float64)
        x1 = torch.as_tensor(x1, dtype=torch.float64)

        self.d = x0.numel()
        self.N = N

        self.register_buffer("x0", x0)
        self.register_buffer("x1", x1)

        if custom_init is not None:
            P_init = torch.as_tensor(
                custom_init,
                dtype=torch.float64,
            )
        else:
            t = torch.linspace(
                0.0,
                1.0,
                N + 2,
                dtype=torch.float64,
            )[1:-1]

            P_init = (
                (1 - t).unsqueeze(1) * x0
                + t.unsqueeze(1) * x1
            )

        self.P_list = nn.ParameterList(
            [
                nn.Parameter(P_init[i].unsqueeze(0))
                for i in range(N)
            ]
        )

    def get_full_path(self):
        P = torch.cat(
            [p for p in self.P_list],
            dim=0,
        )

        return torch.vstack(
            [
                self.x0.unsqueeze(0),
                P,
                self.x1.unsqueeze(0),
            ]
        )

    def forward(self):
        return torch.cat(
            [p for p in self.P_list],
            dim=0,
        )


@torch.no_grad()
def set_model_internal_positions(model, X_NxD):
    X_NxD = X_NxD.detach().to(
        device=model.P_list[0].device,
        dtype=model.P_list[0].dtype,
    )

    for j, p in enumerate(model.P_list):
        p.data = X_NxD[j:j + 1].clone()


def central_difference_tangent(X_full):
    # X_full: (N+2, d) -> 内部 image 的中心差分切向 (N, d)
    return X_full[2:] - X_full[:-2]


# ------------------------------------------------------------
# 3) G-正交投影算子 —— 沿用你的版本
# ------------------------------------------------------------
def G_projectors(partial_s_x, M_seq):
    N, d = partial_s_x.shape

    assert d == 2

    dev = partial_s_x.device
    dt = partial_s_x.dtype

    G_list = []
    tG_list = []

    for i in range(N):
        M_i = M_seq[i]

        # G = M^{-1}
        G_i = torch.linalg.inv(M_i)

        v_i = partial_s_x[i:i + 1, :]
        Gv = v_i @ G_i

        norm2 = (
            torch.sum(
                v_i * Gv,
                dim=1,
                keepdim=True,
            )
            + 1e-16
        )

        norm_G = torch.sqrt(norm2)
        tG_i = v_i / norm_G

        G_list.append(G_i)
        tG_list.append(tG_i)

    # (N, 2, 2)
    G_all = torch.stack(
        G_list,
        dim=0,
    )

    # (N, 2)
    tG_all = torch.cat(
        tG_list,
        dim=0,
    )

    # (N, 2, 1)
    tG_all_3d = tG_all.unsqueeze(2)

    # (N, 2, 1)
    G_tG = torch.bmm(
        G_all,
        tG_all_3d,
    )

    P_parallel_mats = torch.bmm(
        tG_all_3d,
        G_tG.transpose(1, 2),
    )

    denom = torch.sum(
        tG_all * G_tG.squeeze(-1),
        dim=1,
        keepdim=True,
    )

    denom = denom.view(-1, 1, 1) + 1e-16

    P_parallel_mats = (
        P_parallel_mats / denom
    )

    I = (
        torch.eye(
            2,
            device=dev,
            dtype=dt,
        )
        .unsqueeze(0)
        .expand(N, -1, -1)
    )

    P_perp_mats = I - P_parallel_mats

    def P_parallel_G(x):
        return torch.bmm(
            P_parallel_mats,
            x.unsqueeze(-1),
        ).squeeze(-1)

    def P_perp_G(x):
        return torch.bmm(
            P_perp_mats,
            x.unsqueeze(-1),
        ).squeeze(-1)

    return (
        tG_all,
        P_parallel_G,
        P_perp_G,
        P_parallel_mats,
        P_perp_mats,
    )


def P_perp_euclid(x, partial_s_x, eps=1e-16):
    n = torch.sum(
        partial_s_x ** 2,
        axis=1,
        keepdim=True,
    ) ** 0.5

    t = partial_s_x / (n + eps)

    inner = torch.sum(
        t * x,
        axis=1,
        keepdim=True,
    )

    return x - inner * t


# ------------------------------------------------------------
# 4) NEB 切向弹簧重参数化 (论文 Algorithm 2)
#
#    F_sp^i = k_s (l+ - l-) tau_hat_i
#
#    前向欧拉。
#
#    停机:
#       max_i ||F_sp^i|| < tol
#
#    k_s=1 时即:
#       max|l+ - l-| < tol
# ------------------------------------------------------------
def reparam_by_neb(
    X_full,
    k_s=1.0,
    dtau=0.5,
    max_iter=5000,
    tol=1e-3,
    verbose=False,
):
    X = X_full.clone().detach()

    eps = 1e-16
    resid = float("inf")

    for r in range(max_iter):
        # (N+1, d)
        seg = X[1:] - X[:-1]

        # (N+1,)
        seg_len = torch.linalg.norm(
            seg,
            dim=1,
        )

        # (N,)
        l_minus = seg_len[:-1]
        l_plus = seg_len[1:]

        # (N, d)
        tang = X[2:] - X[:-2]

        tang = tang / (
            torch.linalg.norm(
                tang,
                dim=1,
                keepdim=True,
            )
            + eps
        )

        # (N, d)
        F_sp = (
            k_s
            * (l_plus - l_minus).unsqueeze(1)
            * tang
        )

        resid = torch.max(
            torch.linalg.norm(
                F_sp,
                dim=1,
            )
        ).item()

        # Forward Euler
        X[1:-1] = (
            X[1:-1]
            + dtau * F_sp
        )

        if resid < tol:
            if verbose:
                print(
                    f"  [reparam_by_neb] converged @ iter {r}, "
                    f"max|F_sp|={resid:.3e}"
                )
            break

    else:
        if verbose:
            print(
                f"  [reparam_by_neb] hit max_iter={max_iter}, "
                f"max|F_sp|={resid:.3e}"
            )

    return X, resid


# ------------------------------------------------------------
# 5) 沿路径线积分重建自由能剖面
#    仅用于诊断绘图
# ------------------------------------------------------------
def recover_V_along_curve(
    s,
    g,
    phi_prime,
    V_ref=0.0,
):
    dVds = (
        g * phi_prime
    ).sum(
        dim=1,
        keepdim=True,
    )

    ds = s[1:] - s[:-1]

    inc = (
        0.5
        * (dVds[:-1] + dVds[1:])
        * ds
    )

    V = torch.zeros_like(dVds)

    V[0] = V_ref

    V[1:] = (
        V_ref
        + torch.cumsum(
            inc,
            dim=0,
        )
    )

    return (
        V - (torch.min(V) - 1e-6),
        dVds,
    )


# ------------------------------------------------------------
# 6) 绘图辅助 —— 沿用你的定义
# ------------------------------------------------------------
def fig_cos_V_force(
    i,
    s,
    arc,
    cos,
    g,
    v,
    x_pred,
    norm_F_perp,
    folder_path,
    mycase,
    casenum,
):
    force = np.sqrt(
        np.sum(
            g * g,
            axis=1,
        )
    )

    sin_square = 1 - cos ** 2

    fig, axs = plt.subplots(
        1,
        5,
        figsize=(20, 4),
    )

    axs[0].plot(arc, cos, "b.")
    axs[0].set_ylim(-1.1, 1.1)
    axs[0].set_title("cos")
    axs[0].set_box_aspect(1)

    axs[1].plot(arc, sin_square, "b.")
    axs[1].set_ylim(-0.1, 1.1)
    axs[1].set_title("1-cos^2")
    axs[1].set_box_aspect(1)

    axs[2].plot(arc.flatten(), v, "b.")
    axs[2].set_title("Free energy")
    axs[2].set_box_aspect(1)

    axs[3].plot(arc, force, "b.")
    axs[3].set_title("Force")
    axs[3].set_box_aspect(1)

    axs[4].plot(arc, norm_F_perp, "b.")
    axs[4].set_title("norm F_perp")
    axs[4].set_box_aspect(1)

    plt.tight_layout()

    if i % 50 == 0 or i in [1, 100, 200]:
        plt.savefig(
            os.path.join(
                folder_path,
                f"Dot_plot_Iter{i}_{mycase}_{casenum}.eps",
            )
        )

    plt.show()


def fig_loss_batch(
    loss,
    loss_reparam,
    loss_1,
    loss_IVP,
    loss_3,
    loss_EL,
):
    fig, axes = plt.subplots(
        1,
        5,
        figsize=(25, 4),
    )

    axes[0].semilogy(
        loss,
        color="#B41830",
        label="I_perp",
    )

    axes[0].semilogy(
        loss_reparam,
        color="#4F18B4",
        label="reparam resid",
    )

    axes[0].set_title("I_perp & reparam")
    axes[0].legend()

    axes[1].semilogy(
        loss_1,
        color="#B41830",
    )
    axes[1].set_title("fmax")

    axes[2].semilogy(
        loss_IVP,
        color="#B41830",
    )
    axes[2].set_title("I_perp")

    axes[3].semilogy(
        loss_3,
        color="#B41830",
    )
    axes[3].set_title("1-cos^2")

    axes[4].semilogy(
        loss_EL,
        color="#B41830",
    )
    axes[4].set_title("fmax (EL)")

    plt.tight_layout()
    plt.show()


# ============================================================
# 全局固定设置 (不参与调参)
# ============================================================
C1 = [-2.60866606, 2.77661136]
C2 = [-1.3304716, 0.99118795]

N = 20
seed = 41

xlim = (-np.pi, 0)
ylim = (0, np.pi)

REEVALUATE_GRAD_INNER = True

REPARAM_KS = 1.0
REPARAM_DTAU = 1e-3
REPARAM_TOL = 1e-3
REPARAM_MAXITER = 5000
REPARAM_EVERY = 1

base_path = (
    "/Users/wuzhiyou/Code/IVM/"
    "MFEP_ala/amber14sb_vacuo"
)

# 调参时关掉画图
SHOW_FIGS = True


# ============================================================
# 单个配置的完整训练流程
# ============================================================
def run_experiment(
    N_sub,
    alpha_IVP,
    eta_in,
):
    # 约束: N_sub * batches = 300
    batches = 200 // N_sub

    # 复位随机种子
    np.random.seed(seed)
    torch.manual_seed(seed)

    mycase = "alanine"

    casenum = (
        f"MFEP_image_Adam_N{N}"
        f"_Nsub{N_sub}"
        f"_alpha{alpha_IVP}"
        f"_lr{eta_in}"
        f"_NEBreparam"
    )

    folder_path = os.path.join(
        base_path,
        casenum,
    )

    os.makedirs(
        folder_path,
        exist_ok=True,
    )

    figure_path = os.path.join(
        folder_path,
        "figures",
    )

    os.makedirs(
        figure_path,
        exist_ok=True,
    )

    print("=" * 78)

    print(
        f"[GRID] "
        f"N_sub={N_sub}  "
        f"alpha_IVP={alpha_IVP}  "
        f"eta_in={eta_in}  "
        f"batches={batches}  "
        f"(N_sub*batches={N_sub * batches})"
    )

    print("=" * 78)

    # 初始路径: 直线插值
    model = PathPoints(
        C1,
        C2,
        N,
    ).to(device).double()

    # --------------------------------------------------------
    # 诊断序列
    # --------------------------------------------------------

    # 新的、论文定义的 I_perp:
    # sum_i |P_i M_i gradF_i|_{G_i}^2 w_i
    loss_perp_seq = []

    # 原来的指标:
    # mean_i ||P_i M_i gradF_i||_2^2
    loss_perp_euclid_mean_seq = []

    loss_reparam_seq = []
    loss_fmax_seq = []
    loss_cos_seq = []
    index_seq = []

    data = {}

    # --------------------------------------------------------
    # 主循环
    # --------------------------------------------------------
    for i in range(batches + 1):
        # ----------------------------------------------------
        # 冻结当前参考路径
        # ----------------------------------------------------
        X_full = model.get_full_path().detach()
        x_hat = model().detach()

        tangent = central_difference_tangent(
            X_full
        )

        # M(phi_k) 与 grad F(phi_k)
        Mg_ref, M_seq, g_ref = gradient_md_use(
            x_hat
        )

        G_all = torch.stack(
            [
                torch.linalg.inv(M_i)
                for M_i in M_seq
            ],
            dim=0,
        )

        # 冻结的 G-正交投影
        (
            tG_hat,
            P_par_G,
            P_perp_G,
            P_par_mats,
            P_perp_mats,
        ) = G_projectors(
            tangent,
            M_seq,
        )

        # (N, 2, 2)
        M_all = torch.stack(
            M_seq
        )

        RESCALE_MG = False

        def build_perpG_Mg(g_now):
            grad_G_F = torch.bmm(
                M_all,
                g_now.unsqueeze(-1),
            ).squeeze(-1)

            if RESCALE_MG:
                gn = torch.linalg.norm(
                    g_now,
                    dim=1,
                    keepdim=True,
                )

                mn = torch.linalg.norm(
                    grad_G_F,
                    dim=1,
                    keepdim=True,
                )

                grad_G_F = grad_G_F * (
                    gn / (mn + 1e-14)
                )

            return torch.bmm(
                P_perp_mats,
                grad_G_F.unsqueeze(-1),
            ).squeeze(-1)

        # ----------------------------------------------------
        # 诊断量：参考路径处
        # ----------------------------------------------------

        # r_i = P_i grad_G F = P_i M_i grad F
        perpG_Mg = P_perp_G(
            Mg_ref
        )

        # ----------------------------------------------------
        # 原来的 Euclidean norm
        #
        # ||r_i||_2
        # ----------------------------------------------------
        norm_F_perp = torch.sum(
            perpG_Mg ** 2,
            axis=1,
            keepdim=True,
        ) ** 0.5

        fmax = float(
            np.max(
                norm_F_perp
                .detach()
                .cpu()
                .numpy()
            )
        )

        # ----------------------------------------------------
        # 新增：正确的 G-norm
        #
        # |r_i|_{G_i}^2 = r_i^T G_i r_i
        # ----------------------------------------------------
        norm_F_perp_G_sq = torch.einsum(
            "ni,nij,nj->n",
            perpG_Mg,
            G_all,
            perpG_Mg,
        ).clamp_min(0.0)

        # 每个 image 的 G-norm
        norm_F_perp_G = torch.sqrt(
            norm_F_perp_G_sq
        )

        # ----------------------------------------------------
        # 新增：离散 G-弧长 quadrature weight
        #
        # w_i = 1/2 (
        #   |phi_{i+1} - phi_i|_{G_i}
        #   +
        #   |phi_i - phi_{i-1}|_{G_i}
        # )
        # ----------------------------------------------------
        seg_minus = (
            x_hat
            - X_full[:-2]
        )

        seg_plus = (
            X_full[2:]
            - x_hat
        )

        seg_minus_G_sq = torch.einsum(
            "ni,nij,nj->n",
            seg_minus,
            G_all,
            seg_minus,
        ).clamp_min(0.0)

        seg_plus_G_sq = torch.einsum(
            "ni,nij,nj->n",
            seg_plus,
            G_all,
            seg_plus,
        ).clamp_min(0.0)

        seg_minus_G = torch.sqrt(
            seg_minus_G_sq
        )

        seg_plus_G = torch.sqrt(
            seg_plus_G_sq
        )

        node_weights = 0.5 * (
            seg_minus_G
            + seg_plus_G
        )

        # ----------------------------------------------------
        # 新的、正确的论文离散 I_perp:
        #
        # I_perp
        # = sum_i |P_i grad_G F_i|_{G_i}^2 w_i
        # ----------------------------------------------------
        I_perp = torch.sum(
            norm_F_perp_G_sq
            * node_weights
        ).item()

        # ----------------------------------------------------
        # 原来的指标改名并保留:
        #
        # mean_i ||P_i grad_G F_i||_2^2
        # ----------------------------------------------------
        I_perp_euclid_mean = torch.mean(
            norm_F_perp ** 2
        ).item()

        tang_e = tangent / (
            torch.sum(
                tangent ** 2,
                axis=1,
                keepdim=True,
            ) ** 0.5
            + 1e-16
        )

        Mg_perp = P_perp_euclid(
            Mg_ref,
            tang_e,
        )

        # G-度量下的对齐角:
        # <phi', grad_G F>_G = phi'^T grad F
        inner_G = torch.sum(
            tangent * g_ref,
            axis=1,
            keepdim=True,
        )

        tGt = torch.sum(
            tangent
            * torch.bmm(
                G_all,
                tangent.unsqueeze(-1),
            ).squeeze(-1),
            axis=1,
            keepdim=True,
        ) ** 0.5

        gMg = torch.sum(
            g_ref
            * torch.bmm(
                torch.stack(M_seq),
                g_ref.unsqueeze(-1),
            ).squeeze(-1),
            axis=1,
            keepdim=True,
        ) ** 0.5

        cos = inner_G / (
            tGt * gMg + 1e-16
        )

        cos2 = cos ** 2

        # ----------------------------------------------------
        # 自由能剖面
        # ----------------------------------------------------
        s_col = torch.linspace(
            0,
            1,
            N,
            dtype=dtype,
        ).unsqueeze(1)

        ds = 1.0 / (N - 1)

        phi_prime = torch.zeros_like(
            x_hat
        )

        phi_prime[1:-1] = (
            x_hat[2:]
            - x_hat[:-2]
        ) / (2 * ds)

        phi_prime[0] = (
            x_hat[1]
            - x_hat[0]
        ) / ds

        phi_prime[-1] = (
            x_hat[-1]
            - x_hat[-2]
        ) / ds

        V_vals, _ = recover_V_along_curve(
            s_col,
            g_ref,
            phi_prime,
            V_ref=0.0,
        )

        v = (
            V_vals
            .cpu()
            .detach()
            .numpy()
            .flatten()
        )

        # ----------------------------------------------------
        # 内层 Adam 求解
        # ----------------------------------------------------
        delta = nn.Parameter(
            torch.zeros(
                N,
                2,
                dtype=dtype,
                device=device,
            )
        )

        opt_inner = torch.optim.Adam(
            [delta],
            lr=eta_in,
        )

        perpG_Mg_ref = None

        for i_sub in range(N_sub):
            with torch.no_grad():
                current_x = torch.bmm(
                    P_perp_mats,
                    delta.detach().unsqueeze(-1),
                ).squeeze(-1)

                x_eval = x_hat + current_x

            if (
                i_sub == 0
                or not REEVALUATE_GRAD_INNER
            ):
                g = g_ref
            else:
                _, _, g = gradient_md_use(
                    x_eval
                )

            with torch.no_grad():
                perpG_Mg_eval = build_perpG_Mg(
                    g
                )

                if i_sub == 0:
                    perpG_Mg_ref = (
                        perpG_Mg_eval.clone()
                    )

                dI_ddelta = (
                    perpG_Mg_eval
                    + (1.0 / alpha_IVP)
                    * current_x
                )

            opt_inner.zero_grad()

            delta.grad = (
                dI_ddelta
                .detach()
                .clone()
            )

            opt_inner.step()

        # ----------------------------------------------------
        # 提交法向更新
        # ----------------------------------------------------
        with torch.no_grad():
            current_x = torch.bmm(
                P_perp_mats,
                delta.detach().unsqueeze(-1),
            ).squeeze(-1)

            x_new = (
                x_hat
                + current_x
            )

        max_step = float(
            torch.max(
                torch.linalg.norm(
                    current_x,
                    dim=1,
                )
            ).item()
        )

        print(
            f"[outer {i}] "
            f"I_perp={I_perp:.4e}  "
            f"I_perp_euclid_mean={I_perp_euclid_mean:.4e}  "
            f"fmax={fmax:.4e}  "
            f"max_normal_step={max_step:.4e} rad"
        )

        set_model_internal_positions(
            model,
            x_new,
        )

        # ----------------------------------------------------
        # NEB 切向弹簧重参数化
        # ----------------------------------------------------
        reparam_resid = 0.0

        if (
            REPARAM_EVERY > 0
            and i % REPARAM_EVERY == 0
        ):
            X_new_full = (
                model
                .get_full_path()
                .detach()
            )

            X_reparam, reparam_resid = (
                reparam_by_neb(
                    X_new_full,
                    k_s=REPARAM_KS,
                    dtau=REPARAM_DTAU,
                    max_iter=REPARAM_MAXITER,
                    tol=REPARAM_TOL,
                    verbose=(i % 10 == 0),
                )
            )

            set_model_internal_positions(
                model,
                X_reparam[1:-1],
            )

        x_reparam_new = (
            model()
            .detach()
        )

        # ----------------------------------------------------
        # 记录
        # ----------------------------------------------------
        index_seq.append(i)

        # 新的、正确的 I_perp
        loss_perp_seq.append(
            I_perp
        )

        # 原来的 Euclidean mean 指标
        loss_perp_euclid_mean_seq.append(
            I_perp_euclid_mean
        )

        loss_reparam_seq.append(
            reparam_resid
            if reparam_resid > 0
            else 1e-16
        )

        loss_fmax_seq.append(
            fmax
        )

        loss_cos_seq.append(
            float(
                torch.mean(
                    1 - cos2
                ).item()
            )
        )

        # ----------------------------------------------------
        # 可选绘图与模型保存
        # ----------------------------------------------------
        if SHOW_FIGS and (
            i % 1 == 0
            or i in [1, 50, 100, 200]
        ):
            fig_contour(
                i,
                x_hat.cpu().numpy(),
                x_reparam_new.cpu().numpy(),
                None,
                None,
                xlim=xlim,
                ylim=ylim,
            )

            fig_save(
                i * N_sub,
                x_hat.cpu().numpy(),
                x_new.cpu().numpy(),
                x_new.cpu().numpy(),
                x_reparam_new.cpu().numpy(),
                xlim=xlim,
                ylim=ylim,
                force1=-perpG_Mg.cpu().numpy(),
                force2=-g_ref.cpu().numpy(),
                force3=-Mg_perp.cpu().numpy(),
                figure_path=figure_path,
            )

            # fig_cos_V_force(
            #     i,
            #     s_col.cpu().numpy(),
            #     s_col.cpu().numpy(),
            #     cos.cpu().numpy(),
            #     g_ref.cpu().numpy(),
            #     v,
            #     x_hat,
            #     norm_F_perp.cpu().numpy(),
            #     folder_path,
            #     mycase,
            #     casenum,
            # )

            # fig_loss_batch(
            #     np.array(loss_perp_seq),
            #     np.array(loss_reparam_seq),
            #     np.array(loss_fmax_seq),
            #     np.array(loss_perp_seq),
            #     np.array(loss_cos_seq),
            #     np.array(loss_fmax_seq),
            # )

            torch.save(
                model.state_dict(),
                os.path.join(
                    folder_path,
                    f"model_MFEP_"
                    f"{mycase}_{casenum}_{i}.pth",
                ),
            )

            plt.close("all")

        # ----------------------------------------------------
        # 保存全局诊断序列
        # ----------------------------------------------------
        data["index_seq"] = np.asarray(
            index_seq
        )

        # 新的、正确的论文 I_perp
        data["I_perp_seq"] = np.asarray(
            loss_perp_seq
        )

        # 原来的指标，改名后保留
        data["I_perp_euclid_mean_seq"] = np.asarray(
            loss_perp_euclid_mean_seq
        )

        data["fmax_seq"] = np.asarray(
            loss_fmax_seq
        )

        data["reparam_resid_seq"] = np.asarray(
            loss_reparam_seq
        )

        data["cos_seq"] = np.asarray(
            loss_cos_seq
        )

        data[f"path_iter_{i}"] = (
            x_reparam_new
            .cpu()
            .numpy()
        )

        # ----------------------------------------------------
        # delta=0 时每个 image 的准确量
        # ----------------------------------------------------
        data[f"phi_ref_iter_{i}"] = (
            x_hat
            .cpu()
            .numpy()
        )

        data[f"g_ref_iter_{i}"] = (
            g_ref
            .cpu()
            .numpy()
        )

        data[f"perpG_Mg_iter_{i}"] = (
            perpG_Mg_ref
            .cpu()
            .numpy()
        )

        # 原来的每-image Euclidean norm
        data[f"norm_perpG_Mg_iter_{i}"] = (
            torch.linalg.norm(
                perpG_Mg_ref,
                dim=1,
            )
            .cpu()
            .numpy()
        )

        # ----------------------------------------------------
        # 新增：每-image 正确的 G-norm
        # ----------------------------------------------------
        perpG_Mg_ref_G_sq = torch.einsum(
            "ni,nij,nj->n",
            perpG_Mg_ref,
            G_all,
            perpG_Mg_ref,
        ).clamp_min(0.0)

        data[f"norm_perpG_Mg_G_iter_{i}"] = (
            torch.sqrt(
                perpG_Mg_ref_G_sq
            )
            .cpu()
            .numpy()
        )

        # 新增：每-image G-弧长 quadrature weight
        data[f"node_weights_iter_{i}"] = (
            node_weights
            .cpu()
            .numpy()
        )

        np.savez(
            os.path.join(
                folder_path,
                f"{mycase}_{casenum}.npz",
            ),
            **data,
        )

    # --------------------------------------------------------
    # 每个配置结束
    # --------------------------------------------------------
    torch.save(
        model.state_dict(),
        os.path.join(
            folder_path,
            f"model_MFEP_"
            f"{mycase}_{casenum}_final.pth",
        ),
    )

    return {
        "N_sub": N_sub,
        "alpha_IVP": alpha_IVP,
        "eta_in": eta_in,
        "batches": batches,

        # 正确的 I_perp
        "final_I_perp": loss_perp_seq[-1],
        "min_I_perp": float(
            np.min(loss_perp_seq)
        ),

        # 原来的 Euclidean mean 指标
        "final_I_perp_euclid_mean": (
            loss_perp_euclid_mean_seq[-1]
        ),
        "min_I_perp_euclid_mean": float(
            np.min(
                loss_perp_euclid_mean_seq
            )
        ),

        "final_fmax": loss_fmax_seq[-1],
        "final_cos": loss_cos_seq[-1],

        "min_fmax": float(
            np.min(loss_fmax_seq)
        ),
    }


# ============================================================
# 断点续跑辅助函数
# ============================================================
def config_paths(
    N_sub,
    alpha_IVP,
    eta_in,
):
    mycase = "alanine"

    casenum = (
        f"MFEP_image_Adam_N{N}"
        f"_Nsub{N_sub}"
        f"_alpha{alpha_IVP}"
        f"_lr{eta_in}"
        f"_NEBreparam"
    )

    folder_path = os.path.join(
        base_path,
        casenum,
    )

    final_model = os.path.join(
        folder_path,
        f"model_MFEP_"
        f"{mycase}_{casenum}_final.pth",
    )

    npz_file = os.path.join(
        folder_path,
        f"{mycase}_{casenum}.npz",
    )

    return (
        mycase,
        casenum,
        folder_path,
        final_model,
        npz_file,
    )


def is_config_done(
    N_sub,
    alpha_IVP,
    eta_in,
):
    batches = 200 // N_sub

    (
        _,
        _,
        _,
        final_model,
        npz_file,
    ) = config_paths(
        N_sub,
        alpha_IVP,
        eta_in,
    )

    # 1) 最可靠标记
    if os.path.exists(final_model):
        return True

    # 2) npz 兜底
    if os.path.exists(npz_file):
        try:
            with np.load(
                npz_file,
                allow_pickle=True,
            ) as d:
                idx = d["index_seq"]

                if (
                    idx.size > 0
                    and int(np.max(idx)) >= batches
                ):
                    return True

        except Exception:
            return False

    return False


def load_result_from_npz(
    N_sub,
    alpha_IVP,
    eta_in,
):
    """
    从已完成配置的 npz 里恢复诊断量。
    """
    batches = 200 // N_sub

    (
        _,
        _,
        _,
        _,
        npz_file,
    ) = config_paths(
        N_sub,
        alpha_IVP,
        eta_in,
    )

    with np.load(
        npz_file,
        allow_pickle=True,
    ) as d:
        # 新定义的正确 I_perp
        I_perp_seq = np.asarray(
            d["I_perp_seq"],
            dtype=float,
        )

        # 原来的 Euclidean mean 指标
        if "I_perp_euclid_mean_seq" in d.files:
            I_perp_euclid_mean_seq = np.asarray(
                d["I_perp_euclid_mean_seq"],
                dtype=float,
            )
        else:
            # 兼容旧文件：
            # 旧文件中的 I_perp_seq 就是原 Euclidean mean 指标。
            # 注意：这种情况下无法从旧文件恢复新的正确 I_perp。
            I_perp_euclid_mean_seq = I_perp_seq.copy()

        fmax_seq = np.asarray(
            d["fmax_seq"],
            dtype=float,
        )

        cos_seq = np.asarray(
            d["cos_seq"],
            dtype=float,
        )

    return {
        "N_sub": N_sub,
        "alpha_IVP": alpha_IVP,
        "eta_in": eta_in,
        "batches": batches,

        "final_I_perp": float(
            I_perp_seq[-1]
        ),
        "min_I_perp": float(
            np.min(I_perp_seq)
        ),

        "final_I_perp_euclid_mean": float(
            I_perp_euclid_mean_seq[-1]
        ),
        "min_I_perp_euclid_mean": float(
            np.min(
                I_perp_euclid_mean_seq
            )
        ),

        "final_fmax": float(
            fmax_seq[-1]
        ),
        "final_cos": float(
            cos_seq[-1]
        ),
        "min_fmax": float(
            np.min(fmax_seq)
        ),
    }


# ============================================================
# 网格搜索
# ============================================================
GRID_N_SUB = [4, 5, 10]
GRID_ALPHA = [0.1, 1, 10]
GRID_ETA = [0.03, 0.01, 0.003, 0.001]

FORCE_RERUN = True


# ------------------------------------------------------------
# 扫描配置
# ------------------------------------------------------------
all_configs = [
    (ns, al, et)
    for ns in GRID_N_SUB
    for al in GRID_ALPHA
    for et in GRID_ETA
]

done_flags = {
    cfg: is_config_done(*cfg)
    for cfg in all_configs
}

n_done = sum(
    done_flags.values()
)

print("=" * 78)

print(
    f"[RESUME] 共 {len(all_configs)} 个配置; "
    f"检测到已完成 {n_done} 个, "
    f"待跑 {len(all_configs) - n_done} 个"
    + (
        "  (FORCE_RERUN=True, 将全部重跑)"
        if FORCE_RERUN
        else ""
    )
)

for cfg in all_configs:
    ns, al, et = cfg

    tag = (
        "DONE  跳过"
        if (
            done_flags[cfg]
            and not FORCE_RERUN
        )
        else "TODO  待跑"
    )

    print(
        f"    [{tag}]  "
        f"N_sub={ns:>2}  "
        f"alpha={al:>4}  "
        f"eta={et}"
    )

print("=" * 78)


# ------------------------------------------------------------
# 正式执行
# ------------------------------------------------------------
summary = []

for cfg in all_configs:
    N_sub, alpha_IVP, eta_in = cfg

    if (
        done_flags[cfg]
        and not FORCE_RERUN
    ):
        print(
            f"[SKIP] "
            f"N_sub={N_sub} "
            f"alpha={alpha_IVP} "
            f"eta={eta_in} "
            f"已完成, 直接读取旧结果"
        )

        try:
            summary.append(
                load_result_from_npz(
                    N_sub,
                    alpha_IVP,
                    eta_in,
                )
            )

        except Exception as e:
            print(
                "       "
                f"(读取旧结果失败, "
                f"汇总里将缺此项: {e})"
            )

        continue

    res = run_experiment(
        N_sub,
        alpha_IVP,
        eta_in,
    )

    summary.append(res)


# ============================================================
# 汇总
# ============================================================
print("\n" + "=" * 78)
print("GRID SEARCH SUMMARY  (按 final_fmax 升序)")
print("=" * 78)

summary_sorted = sorted(
    summary,
    key=lambda r: r["final_fmax"],
)

for r in summary_sorted:
    print(
        f"N_sub={r['N_sub']:>2}  "
        f"alpha={r['alpha_IVP']:>4}  "
        f"eta={r['eta_in']:<6}  "
        f"| final_I_perp={r['final_I_perp']:.3e}  "
        f"final_I_perp_euclid_mean="
        f"{r['final_I_perp_euclid_mean']:.3e}  "
        f"final_fmax={r['final_fmax']:.3e}  "
        f"final_1-cos2={r['final_cos']:.3e}  "
        f"min_fmax={r['min_fmax']:.3e}"
    )


np.savez(
    os.path.join(
        base_path,
        "GRID_SUMMARY.npz",
    ),
    summary=np.array(
        summary,
        dtype=object,
    ),
)